# C6 · sustracción de halo LPM — notebook de análisis (`debug`)

**Objeto:** ROXs42Bb  |  **Run:** `ROXs42Bb_realigned`  |  **Spec:** [`docs/spec_C6_codex_lpm_subtraction.md`](../../../docs/spec_C6_codex_lpm_subtraction.md)

**LPM** (Julo+25 App. A.4) parte de la misma idea que el SGF pero **no filtra**: modela cada spaxel como el espectro de referencia **modulado por un polinomio de Legendre** de grado bajo,

> `ŝ_xy(λ) = Σ_k β_k · P_k(λ̃) · ŝ(λ)`

y ajusta los `β_k` por mínimos cuadrados **con las líneas de ciencia enmascaradas**. Esa máscara es la diferencia clave con el SGF: como la línea no entra en el ajuste, el modelo no puede aprenderla, y por eso **el LPM preserva la línea** (su chequeo `v2_line_preservation_ok` exige recuperar ≥90% de una línea inyectada).

El esqueleto de la etapa son cuatro pasos: **elegir los spaxels de referencia**, **construir el espectro estelar de referencia**, **restar el halo** (lo propio de cada método) y **extraer una apertura box3** del cubo residual — esta última con la misma maquinaria que C2, controles y `apcorr` incluidos.


In [ ]:
import json, sys
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
# Resolución de las figuras EN PANTALLA. `savefig` guarda a 300 dpi, pero
# lo que se ve dentro del notebook lo fija el backend inline, que va a 100
# dpi por defecto y sale borroso. `retina` dobla los píxeles sin cambiar el
# tamaño aparente; fuera de IPython no hace nada y queda el rcParam.
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['savefig.dpi'] = 200
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
from matplotlib.colors import LogNorm

_here = Path.cwd()
ROOT = next(p for p in (_here, *_here.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)
print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)


## 1 · Perillas

Del **config resuelto de la etapa**: rellena defaults que el run no escribe. Cambia lo que quieras debajo de la lectura.


In [ ]:
from musepipe.stages.stage_x05_lpm import stage_x05_config_from_run as _cfg_from_run

# `project_root=ROOT`: musepipe resuelve rutas contra el cwd, que en un
# notebook es su propia carpeta, no la raíz del repo.
X0 = _cfg_from_run(RUN_ID, project_root=ROOT)
# OJO: la perilla del método NO lleva prefijo de etapa (`lpm_degree`, no `x05_*`).
LPM_DEGREE = int(X0.get('lpm_degree', 4))   # grado del polinomio de Legendre
FLUX_LO = float(X0.get('halosub_flux_mask_lo', 0.10))   # percentil bajo de la máscara
FLUX_HI = float(X0.get('halosub_flux_mask_hi', 0.90))   # percentil alto
EXCLUDE_RADIUS_PX = float(X0.get('halosub_exclude_radius_px', 3.0))
N_CONTROLS = int(X0.get('x05_control_apertures', 8))
EXCLUDE_ANGLE_DEG = float(X0.get('x05_control_exclude_angle_deg', 25.0))
APCORR_MODE = X0.get('x05_aperture_correction', 'auto')
ERROR_MODE = X0.get('x05_error_mode', 'auto')
ANNULUS = X0.get('x05_annulus_bkg_px')
BAD_WINDOWS_A = X0.get('x05_bad_windows_A', [])

# ---- a partir de aquí, cambia lo que quieras probar ----

print('máscara de referencia: percentiles', FLUX_LO, '-', FLUX_HI,
      '| excluye', EXCLUDE_RADIUS_PX, 'px alrededor del compañero')
print('extracción: controles', N_CONTROLS, '| apcorr', APCORR_MODE,
      '| anillo', ANNULUS)


## 2 · Entradas

El cubo de B2 (la etapa trabaja **por exposición** si el run las tiene; aquí se usa el stack, que es lo que hay en estos runs) y las posiciones de B3.


In [ ]:
qc_b3 = json.loads((SD / 'stage01c_qc.json').read_text(encoding='utf-8'))
OBJECT_YX = tuple(float(v) for v in qc_b3['companion']['pos_yx'])
STAR_YX   = tuple(float(v) for v in qc_b3['primary']['pos_yx'])
psf_path = SD / 'psf_model.json'
PSF_MODEL = json.loads(psf_path.read_text(encoding='utf-8')) if psf_path.exists() else None

CUBE_PATH = SD / 'stage02_xcorr_cube_stack.fits'
with fits.open(CUBE_PATH) as h:
    CUBE = np.asarray(h['CUBES'].data, dtype=float)
    WAVE = np.asarray(h['WAVELENGTH'].data, dtype=float)
    STAT_CUBE = np.asarray(h['STAT'].data, dtype=float) if 'STAT' in h else None
    _stack_bunit = str(h[0].header.get('BUNIT', '')
                       or h['CUBES'].header.get('BUNIT', '')) or None
if CUBE.ndim == 4:
    CUBE = CUBE[0]
if STAT_CUBE is not None and STAT_CUBE.ndim == 4:
    STAT_CUBE = STAT_CUBE[0]
# La unidad, con la regla de la cadena: el stack de B2 no la declara y
# `resolve_bunit` cae al cubo de entrada del run.
from musepipe.io import resolve_bunit
BUNIT = resolve_bunit(X0, stack_bunit=_stack_bunit)
UNIDAD = BUNIT or 'sin unidad declarada'

qc00 = json.loads((SD / 'stage00q_qc.json').read_text(encoding='utf-8'))
qc01 = json.loads((SD / 'stage01_qc.json').read_text(encoding='utf-8'))
m5 = qc00.get('m5_stat', {})
STAT_FACTOR = float(X0.get('x05_stat_factor_box3', m5.get('factor_box3_median', 1.0)) or 1.0)
COV_FACTOR  = float(X0.get('x05_covariance_factor_box3',
                            qc01.get('stat', {}).get('covariance_factor_box3', 1.0)) or 1.0)
STAT_STATUS = str(X0.get('x05_stat_status', m5.get('status', 'unknown')))
print('cubo     :', CUBE.shape, '| compañero', [round(v, 1) for v in OBJECT_YX],
      '| primaria', [round(v, 1) for v in STAR_YX])
print(f'STAT     : factor={STAT_FACTOR:.3f} covarianza={COV_FACTOR:.3f} estado={STAT_STATUS}')


## 3 · Las funciones copiadas de `musepipe`

- `finite_values` — de `musepipe/stats.py`
- `robust_sigma` — de `musepipe/stats.py`
- `robust_sigma_axis0` — de `musepipe/stats.py`
- `angular_separation_deg` — de `musepipe/apertures.py`
- `aperture_weights` — de `musepipe/apertures.py`
- `same_radius_control_positions` — de `musepipe/apertures.py`
- `_as_cube` — de `musepipe/extraction/aperture.py`
- `_npix_eff` — de `musepipe/extraction/aperture.py`
- `aperture_spectrum` — de `musepipe/extraction/aperture.py`
- `annulus_background_spectrum` — de `musepipe/extraction/aperture.py`
- `aperture_stat_error` — de `musepipe/extraction/aperture.py`
- `control_aperture_spectra` — de `musepipe/extraction/aperture.py`
- `_flag_window` — de `musepipe/extraction/aperture.py`
- `channel_flags` — de `musepipe/extraction/aperture.py`
- `aperture_correction_from_psf` — de `musepipe/extraction/aperture.py`
- `select_reference_spaxels` — de `musepipe/halosub.py`
- `reference_spectrum` — de `musepipe/halosub.py`
- `safe_reference` — de `musepipe/halosub.py`
- `fill_nan_along_axis0` — de `musepipe/halosub.py`
- `LpmResult` — de `musepipe/halosub.py`
- `lpm_design_matrix` — de `musepipe/halosub.py`
- `lpm_fit_mask` — de `musepipe/halosub.py`
- `lpm_subtract` — de `musepipe/halosub.py`
- `lpm_coefficient_energy_share` — de `musepipe/halosub.py`


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from dataclasses import dataclass
from musepipe.psf import evaluate_psf_model
from musepipe.spectral import STANDARD_LINE_WINDOWS_A
from musepipe.spectral import standard_line_free_mask
from numpy.polynomial import legendre as npleg
from typing import Sequence
import math
import numpy as np
import warnings
# `evaluate_psf_model` (C1) y el catálogo de líneas se importan arriba:
# no son lo que se ajusta aquí.

FLAG_BAD_WINDOW = 1
FLAG_SKYLINE = 2
FLAG_INTERPOLATED = 4
FLAG_CLIPPED = 8
DEFAULT_FLUX_MASK_LO = 0.01
DEFAULT_FLUX_MASK_HI = 0.1
DEFAULT_LPM_DEGREE = 4


def finite_values(values) -> np.ndarray:
    """Return finite values as a float64 1D array."""

    arr = np.asarray(values, dtype=np.float64)
    return arr[np.isfinite(arr)]


def robust_sigma(values) -> float:
    """Robust 1D sigma estimate using MAD with std fallback."""

    vals = finite_values(values)
    if vals.size == 0:
        return np.nan
    med = np.nanmedian(vals)
    mad = np.nanmedian(np.abs(vals - med))
    sigma = 1.4826 * mad
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = np.nanstd(vals)
    return float(sigma)


def robust_sigma_axis0(values) -> np.ndarray:
    """Robust sigma along axis 0 using MAD with std fallback per column."""

    arr = np.asarray(values, dtype=np.float64)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        med = np.nanmedian(arr, axis=0)
        mad = np.nanmedian(np.abs(arr - med[None, :]), axis=0)
        sigma = 1.4826 * mad
        std = np.nanstd(arr, axis=0)
    bad = ~np.isfinite(sigma) | (sigma <= 0)
    sigma[bad] = std[bad]
    return sigma


def angular_separation_deg(a, b) -> float:
    """Smallest angular separation between two angles in radians, in degrees."""

    return abs(math.degrees(math.atan2(math.sin(a - b), math.cos(a - b))))


def aperture_weights(ny, nx, center_yx, aperture) -> np.ndarray:
    """Build a 2D aperture-weight image."""

    y0, x0 = map(float, center_yx)
    yy, xx = np.mgrid[:ny, :nx]
    rr2 = (yy - y0) ** 2 + (xx - x0) ** 2
    weights = np.zeros((ny, nx), dtype=np.float64)
    kind = aperture["kind"]

    if kind == "pixel":
        y = int(round(y0))
        x = int(round(x0))
        if 0 <= y < ny and 0 <= x < nx:
            weights[y, x] = 1.0
    elif kind == "box":
        size = int(aperture.get("size", 3))
        half = size // 2
        y = int(round(y0))
        x = int(round(x0))
        y1 = max(0, y - half)
        y2 = min(ny, y + half + 1)
        x1 = max(0, x - half)
        x2 = min(nx, x + half + 1)
        weights[y1:y2, x1:x2] = 1.0
    elif kind == "circle":
        radius = float(aperture["radius_px"])
        weights[rr2 <= radius**2] = 1.0
    elif kind == "gaussian":
        sigma = float(aperture["sigma_px"])
        radius = float(aperture.get("radius_px", 3.0 * sigma))
        mask = rr2 <= radius**2
        weights[mask] = np.exp(-0.5 * rr2[mask] / sigma**2)
    else:
        raise ValueError(f"Unknown aperture kind: {kind}")
    return weights


def same_radius_control_positions(
    object_yx,
    star_yx,
    ny,
    nx,
    n_positions=8,
    exclude_angle_deg=25.0,
    margin_px=4,
):
    """Return integer control positions at the same star-object radius."""

    oy, ox = map(float, object_yx)
    sy, sx = map(float, star_yx)
    dy = oy - sy
    dx = ox - sx
    radius = math.hypot(dy, dx)
    theta0 = math.atan2(dy, dx)

    controls = []
    for k in range(int(n_positions)):
        theta = theta0 + 2.0 * math.pi * k / float(n_positions)
        if angular_separation_deg(theta, theta0) < exclude_angle_deg:
            continue
        y = int(round(sy + radius * math.sin(theta)))
        x = int(round(sx + radius * math.cos(theta)))
        if margin_px <= y < ny - margin_px and margin_px <= x < nx - margin_px:
            controls.append((y, x))
    return controls


def _as_cube(cube_zyx, name="cube") -> np.ndarray:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected {name} with shape (nz,ny,nx), got {cube.shape}.")
    return cube


def _npix_eff(cube_zyx: np.ndarray, weights: np.ndarray) -> np.ndarray:
    valid = np.isfinite(cube_zyx) & (weights[None, :, :] > 0)
    sumw = np.sum(weights[None, :, :] * valid, axis=(1, 2))
    sumw2 = np.sum((weights[None, :, :] ** 2) * valid, axis=(1, 2))
    out = np.full(cube_zyx.shape[0], np.nan, dtype=np.float64)
    good = sumw2 > 0
    out[good] = (sumw[good] ** 2) / sumw2[good]
    return out


def aperture_spectrum(cube_zyx, center_yx, aperture: dict) -> tuple[np.ndarray, np.ndarray]:
    """Return weighted-sum spectrum and per-channel effective pixel count."""

    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    weights = aperture_weights(ny, nx, center_yx, aperture)
    weighted = cube * weights[None, :, :]
    with np.errstate(invalid="ignore"):
        flux = np.nansum(weighted, axis=(1, 2)).astype(np.float64)
    npix_eff = _npix_eff(cube, weights)
    flux[~np.isfinite(npix_eff)] = np.nan
    return flux, npix_eff


def annulus_background_spectrum(cube_zyx, center_yx, r_in, r_out, *, exclude_yx=None, exclude_radius=0.0):
    """Per-channel local background = median of a source-free annulus.

    Used for the wings-intact aperture-correction path: subtracting a distant
    annulus (rather than a local surface, stage04b) preserves the companion's
    PSF wings so the PSF growth-curve aperture correction stays self-consistent
    (box3<box5). Excludes a region around ``exclude_yx`` (the primary)."""

    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    yy, xx = np.mgrid[0:ny, 0:nx]
    r = np.hypot(yy - float(center_yx[0]), xx - float(center_yx[1]))
    mask = (r >= float(r_in)) & (r <= float(r_out))
    if exclude_yx is not None and float(exclude_radius) > 0:
        mask &= np.hypot(yy - float(exclude_yx[0]), xx - float(exclude_yx[1])) > float(exclude_radius)
    if not mask.any():
        return np.zeros(cube.shape[0], dtype=np.float64)
    vals = cube[:, mask]
    with np.errstate(all="ignore"):
        return np.nanmedian(vals, axis=1).astype(np.float64)


def aperture_stat_error(
    stat_zyx,
    center_yx,
    aperture: dict,
    *,
    stat_factor: float = 1.0,
    covariance_factor: float = 1.0,
) -> np.ndarray:
    """Propagate a variance cube through the same aperture weights."""

    stat = _as_cube(stat_zyx, name="stat")
    _, ny, nx = stat.shape
    weights = aperture_weights(ny, nx, center_yx, aperture)
    valid = np.isfinite(stat) & (weights[None, :, :] > 0)
    variance = np.nansum(stat * (weights[None, :, :] ** 2), axis=(1, 2))
    variance[np.sum(valid, axis=(1, 2)) == 0] = np.nan
    factor = float(stat_factor) * float(covariance_factor)
    if not np.isfinite(factor) or factor <= 0:
        factor = 1.0
    variance *= factor
    return np.sqrt(np.clip(variance, 0.0, np.inf)).astype(np.float64)


def control_aperture_spectra(
    cube_zyx,
    object_yx,
    star_yx,
    aperture: dict,
    *,
    n_controls: int = 8,
    exclude_angle_deg: float = 25.0,
    margin_px: int | None = None,
) -> tuple[list[tuple[int, int]], np.ndarray]:
    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    if margin_px is None:
        if str(aperture.get("kind", "box")) == "box":
            margin_px = int(aperture.get("size", 3)) // 2 + 1
        else:
            margin_px = int(math.ceil(float(aperture.get("radius_px", 3.0)))) + 1
    controls = same_radius_control_positions(
        object_yx,
        star_yx,
        ny,
        nx,
        n_positions=int(n_controls),
        exclude_angle_deg=float(exclude_angle_deg),
        margin_px=int(margin_px),
    )
    spectra = []
    npix = []
    for yx in controls:
        flux, npix_eff = aperture_spectrum(cube, yx, aperture)
        spectra.append(flux)
        npix.append(npix_eff)
    if not spectra:
        empty = np.empty((0, cube.shape[0]), dtype=np.float64)
        return controls, empty, empty.copy()
    return controls, np.asarray(spectra, dtype=np.float64), np.asarray(npix, dtype=np.float64)


def _flag_window(wave_A: np.ndarray, windows_A: Sequence[Sequence[float]], bit: int, flags: np.ndarray) -> None:
    for window in windows_A or ():
        if window is None or len(window) != 2:
            continue
        lo, hi = window
        if lo is None or hi is None:
            continue
        flags[(wave_A >= float(lo)) & (wave_A <= float(hi))] |= int(bit)


def channel_flags(
    wave_A,
    *,
    bad_windows_A: Sequence[Sequence[float]] = (),
    skyline_windows_A: Sequence[Sequence[float]] = (),
    interpolated_windows_A: Sequence[Sequence[float]] = (),
    clipped_mask=None,
    good_mask=None,
    bad_mask=None,
) -> np.ndarray:
    wave = np.asarray(wave_A, dtype=np.float64)
    flags = np.zeros(wave.size, dtype=np.int32)
    _flag_window(wave, bad_windows_A, FLAG_BAD_WINDOW, flags)
    _flag_window(wave, skyline_windows_A, FLAG_SKYLINE, flags)
    _flag_window(wave, interpolated_windows_A, FLAG_INTERPOLATED, flags)
    if good_mask is not None:
        flags[~np.asarray(good_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if bad_mask is not None:
        flags[np.asarray(bad_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if clipped_mask is not None:
        flags[np.asarray(clipped_mask, dtype=bool)] |= FLAG_CLIPPED
    return flags


def aperture_correction_from_psf(
    wave_A,
    aperture: dict,
    psf_model: dict | None,
    *,
    center_yx=(0.0, 0.0),
    correction_mode: str = "auto",
    growth_curve=None,
) -> tuple[np.ndarray, str, float]:
    """Return wavelength-dependent aperture correction from a C1 PSF model.

    By default the PSF is normalized to 1 inside ``norm_radius_px`` (25 px =
    0.63" in NFM), so the recovered "total flux" is really *the flux inside that
    radius*. Measured on the A2-size cube, 40-55% of the modelled light lies
    outside it, and the missing factor is chromatic (~2.5 blue, ~1.9 red), so it
    does not cancel -- it tilts the continuum.

    Passing ``growth_curve`` (A2's ``growth_curve`` QC block) multiplies the
    correction by the empirically measured ``F_total / F(<=norm_radius)`` and
    switches the convention to genuine total flux. A2 measures it; the caller
    decides -- see ``x01_flux_convention``.
    """

    wave = np.asarray(wave_A, dtype=np.float64)
    mode = str(correction_mode or "auto").lower()
    if mode in {"none", "off", "false"}:
        return np.ones(wave.size, dtype=np.float64), "none", 0.0
    if psf_model is None:
        if mode in {"auto", "optional"}:
            return np.ones(wave.size, dtype=np.float64), "none", 0.0
        raise RuntimeError("Aperture correction requested but no psf_model was supplied.")

    norm_radius = float(psf_model.get("norm_radius_px", 25.0))
    half = int(math.ceil(norm_radius))
    frac_y = float(center_yx[0]) - round(float(center_yx[0]))
    frac_x = float(center_yx[1]) - round(float(center_yx[1]))
    source_center = (half + frac_y, half + frac_x)
    yy, xx = np.indices((2 * half + 1, 2 * half + 1), dtype=np.float64)
    dy = yy - source_center[0]
    dx = xx - source_center[1]
    weights = aperture_weights(2 * half + 1, 2 * half + 1, source_center, aperture)
    fractions = np.empty(wave.size, dtype=np.float64)
    for i, w in enumerate(wave):
        psf = evaluate_psf_model(psf_model, float(w), dy, dx)
        frac = float(np.nansum(psf * weights))
        if not np.isfinite(frac) or frac <= 0:
            raise RuntimeError(f"Invalid aperture PSF fraction at wave={w}.")
        fractions[i] = frac
    apcorr = (1.0 / fractions).astype(np.float64)
    if growth_curve:
        # Import ABSOLUTO y dentro de la funcion: esta funcion se COPIA
        # literalmente dentro de los notebooks de `debug/`, donde un import
        # relativo (`from ..growth_curve`) revienta con ImportError por no
        # haber paquete padre. El absoluto funciona en los dos sitios.
        from musepipe.growth_curve import factor_at_wavelengths

        factor = np.asarray(factor_at_wavelengths(growth_curve, wave), dtype=np.float64)
        if not np.all(np.isfinite(factor)) or np.any(factor <= 0):
            raise RuntimeError("Growth-curve total-flux factor is not finite and positive.")
        return apcorr * factor, "psf_growth_curve+empirical_total", norm_radius
    return apcorr, "psf_growth_curve", norm_radius


def select_reference_spaxels(
    cube_zyx,
    *,
    flux_lo_frac=DEFAULT_FLUX_MASK_LO,
    flux_hi_frac=DEFAULT_FLUX_MASK_HI,
    wave_mask=None,
    exclude_yx=None,
    exclude_radius_px=0.0,
):
    """Spatial mask of spaxels used for the stellar reference spectrum.

    Follows the paper's Table 1: spaxels with integrated flux below
    ``flux_lo_frac * Fmax`` (too noisy) or above ``flux_hi_frac * Fmax``
    (most distorted by the pipeline / star core) are excluded.

    ``exclude_yx`` optionally removes discs of ``exclude_radius_px`` around
    known sources (e.g. the companion), so its flux never contaminates the
    reference.

    Returns ``(keep_mask_2d, qc_dict)``.
    """

    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube (nz, ny, nx), got shape {cube.shape}.")
    if not 0.0 <= float(flux_lo_frac) < float(flux_hi_frac) <= 1.0:
        raise ValueError("Need 0 <= flux_lo_frac < flux_hi_frac <= 1.")

    nz = cube.shape[0]
    if wave_mask is None:
        wmask = np.ones(nz, dtype=bool)
    else:
        wmask = np.asarray(wave_mask, dtype=bool)
        if wmask.shape != (nz,):
            raise ValueError("wave_mask must match the spectral axis.")

    with np.errstate(all="ignore"):
        total = np.nansum(cube[wmask], axis=0)
        n_finite = np.sum(np.isfinite(cube[wmask]), axis=0)
    valid = n_finite > 0
    total = np.where(valid, total, np.nan)

    fmax = np.nanmax(total) if np.any(valid) else np.nan
    if not np.isfinite(fmax) or fmax <= 0:
        raise RuntimeError("Cannot select reference spaxels: no positive integrated flux.")

    keep = valid & (total > float(flux_lo_frac) * fmax) & (total < float(flux_hi_frac) * fmax)

    if exclude_yx:
        ny, nx = cube.shape[1:]
        yy, xx = np.indices((ny, nx), dtype=np.float64)
        for y0, x0 in exclude_yx:
            rr2 = (yy - float(y0)) ** 2 + (xx - float(x0)) ** 2
            keep &= rr2 > float(exclude_radius_px) ** 2

    qc = {
        "n_spaxels_kept": int(np.sum(keep)),
        "n_spaxels_valid": int(np.sum(valid)),
        "flux_lo_frac": float(flux_lo_frac),
        "flux_hi_frac": float(flux_hi_frac),
        "fmax": float(fmax),
    }
    if qc["n_spaxels_kept"] == 0:
        raise RuntimeError("Reference spaxel selection kept 0 spaxels; check flux mask fractions.")
    return keep, qc


def reference_spectrum(cube_zyx, keep_mask=None):
    """Median stellar reference spectrum over the selected spaxels.

    Median (not mean/sum) for robustness, as in the paper App. A.2; the
    flux-conservation constant is irrelevant because both methods estimate a
    multiplicative deformation per spaxel.
    """

    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube (nz, ny, nx), got shape {cube.shape}.")
    if keep_mask is None:
        spax = cube.reshape(cube.shape[0], -1)
    else:
        keep = np.asarray(keep_mask, dtype=bool)
        if keep.shape != cube.shape[1:]:
            raise ValueError("keep_mask must match the spatial axes.")
        spax = cube[:, keep]
    with np.errstate(all="ignore"):
        return np.nanmedian(spax, axis=1)


def safe_reference(s_hat, eps_frac=1e-6):
    """Reference with near-zero / non-finite channels set to NaN (safe divisor)."""

    ref = np.asarray(s_hat, dtype=np.float64).copy()
    scale = np.nanmedian(np.abs(ref))
    if not np.isfinite(scale) or scale <= 0:
        raise RuntimeError("Reference spectrum has no finite scale.")
    bad = ~np.isfinite(ref) | (np.abs(ref) < float(eps_frac) * scale)
    ref[bad] = np.nan
    return ref


def fill_nan_along_axis0(values):
    """Linear interpolation of NaNs along axis 0 (edges: nearest finite).

    Returns ``(filled, finite_mask)``. Columns without any finite value stay
    NaN in ``filled``.
    """

    arr = np.asarray(values, dtype=np.float64)
    flat = arr.reshape(arr.shape[0], -1)
    finite = np.isfinite(flat)
    filled = flat.copy()
    idx = np.arange(flat.shape[0], dtype=np.float64)
    needs = np.any(~finite, axis=0) & np.any(finite, axis=0)
    for j in np.where(needs)[0]:
        good = finite[:, j]
        filled[~good, j] = np.interp(idx[~good], idx[good], flat[good, j])
    return filled.reshape(arr.shape), finite.reshape(arr.shape)


@dataclass(frozen=True)
class LpmResult:
    stellar_cube: np.ndarray
    residual_cube: np.ndarray
    coeffs: np.ndarray  # (degree+1, ny, nx)
    ref_used: np.ndarray
    fit_mask: np.ndarray  # channels used in the fit (common mask)
    degree: int
    condition_number: float
    n_slow_spaxels: int


def lpm_design_matrix(wave_A, s_hat, degree=DEFAULT_LPM_DEGREE):
    """Design matrix M (paper Eq. A.13): Legendre polynomials modulating s_hat.

    Column k is ``P_k(lambda_scaled) * s_hat``; wavelengths are mapped to
    [-1, 1] over the finite channels so the basis is (near) orthogonal and
    the Gram matrix stays well conditioned (paper App. A.4).
    """

    wave = np.asarray(wave_A, dtype=np.float64)
    ref = np.asarray(s_hat, dtype=np.float64)
    if wave.shape != ref.shape:
        raise ValueError("wave_A and s_hat must have the same shape.")
    degree = int(degree)
    if degree < 0:
        raise ValueError("degree must be >= 0.")

    good = np.isfinite(wave)
    if not np.any(good):
        raise RuntimeError("No finite wavelengths for the LPM basis.")
    lo, hi = float(np.min(wave[good])), float(np.max(wave[good]))
    if hi <= lo:
        raise RuntimeError("Degenerate wavelength range for the LPM basis.")
    x = 2.0 * (wave - lo) / (hi - lo) - 1.0
    x = np.where(good, x, 0.0)

    vander = npleg.legvander(x, degree)  # (nz, degree+1)
    return vander * ref[:, None]


def lpm_fit_mask(wave_A, s_hat, *, extra_mask=None, line_windows_A=STANDARD_LINE_WINDOWS_A):
    """Channels entering the LPM fit: finite reference, outside line windows.

    ``line_windows_A`` follows ``STANDARD_LINE_WINDOWS_A`` (Halpha/Hbeta/OI)
    by default; stages pass the per-target list frozen in their spec. This is
    the paper's key ingredient against self-subtraction: the lines never
    constrain the stellar model, which interpolates smoothly across them.
    """

    ref = safe_reference(s_hat)
    mask = standard_line_free_mask(wave_A, base_mask=np.isfinite(ref), line_windows_A=line_windows_A)
    if extra_mask is not None:
        mask &= np.asarray(extra_mask, dtype=bool)
    return mask


def lpm_subtract(
    cube_zyx,
    wave_A,
    s_hat,
    *,
    degree=DEFAULT_LPM_DEGREE,
    fit_mask=None,
    line_windows_A=STANDARD_LINE_WINDOWS_A,
    min_channel_finite_frac=0.75,
):
    """LPM halo subtraction of one cube (paper Eq. A.14/A.15).

    The pseudo-inverse of the masked design matrix is computed once and
    applied to every spaxel (the model is shared; only the data change).
    Channel hygiene (spec C6 v1.1): channels finite in less than
    ``min_channel_finite_frac`` of the spaxels are dropped from the fit mask
    (half-bad columns would push every spaxel to the slow path). Spaxels with
    additional non-finite channels inside the fit mask fall back to a
    per-spaxel least squares (slow path, counted in the result).
    """

    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube (nz, ny, nx), got shape {cube.shape}.")
    nz, ny, nx = cube.shape

    ref = safe_reference(s_hat)
    if fit_mask is None:
        fit_mask = lpm_fit_mask(wave_A, ref, line_windows_A=line_windows_A)
    else:
        fit_mask = np.asarray(fit_mask, dtype=bool) & np.isfinite(ref)
    if min_channel_finite_frac is not None and ny * nx > 1:
        finite_all = np.isfinite(cube.reshape(nz, ny * nx))
        spax_has_data = np.any(finite_all, axis=0)
        if np.any(spax_has_data):
            channel_frac = finite_all[:, spax_has_data].mean(axis=1)
            fit_mask = fit_mask & (channel_frac >= float(min_channel_finite_frac))
    n_params = int(degree) + 1
    if int(np.sum(fit_mask)) <= n_params:
        raise RuntimeError(
            f"LPM fit mask keeps {int(np.sum(fit_mask))} channels for {n_params} parameters."
        )

    design = lpm_design_matrix(wave_A, ref, degree=degree)
    m_fit = design[fit_mask]
    pinv = np.linalg.pinv(m_fit)
    condition = float(np.linalg.cond(m_fit))

    data = cube.reshape(nz, ny * nx)
    data_fit = data[fit_mask]
    finite_fit = np.isfinite(data_fit)
    fast = np.all(finite_fit, axis=0)

    coeffs = np.full((n_params, ny * nx), np.nan, dtype=np.float64)
    if np.any(fast):
        coeffs[:, fast] = pinv @ data_fit[:, fast]

    slow_idx = np.where(~fast & np.any(finite_fit, axis=0))[0]
    for j in slow_idx:
        good = finite_fit[:, j]
        if int(np.sum(good)) <= n_params:
            continue
        beta, *_ = np.linalg.lstsq(m_fit[good], data_fit[good, j], rcond=None)
        coeffs[:, j] = beta

    stellar = design @ coeffs  # (nz, ny*nx); NaN coeffs propagate
    stellar = stellar.reshape(nz, ny, nx)
    residual = cube - stellar
    return LpmResult(
        stellar_cube=stellar,
        residual_cube=residual,
        coeffs=coeffs.reshape(n_params, ny, nx),
        ref_used=ref,
        fit_mask=fit_mask,
        degree=int(degree),
        condition_number=condition,
        n_slow_spaxels=int(slow_idx.size),
    )


def lpm_coefficient_energy_share(coeffs):
    """Per-degree energy share of the LPM coefficients (paper Fig. 8).

    Degree 0 carries the total flux and is excluded (as in the paper); the
    remaining energies are medians over the field of the squared coefficients,
    normalized to sum 1. Used by the C6 QC ``lpm_degree_check``.
    """

    arr = np.asarray(coeffs, dtype=np.float64)
    if arr.ndim != 3 or arr.shape[0] < 2:
        raise ValueError("coeffs must be (degree+1, ny, nx) with degree >= 1.")
    with np.errstate(all="ignore"):
        energy = np.array([np.nanmedian(arr[k] ** 2) for k in range(1, arr.shape[0])])
    total = np.nansum(energy)
    if not np.isfinite(total) or total <= 0:
        raise RuntimeError("Degenerate coefficient energies.")
    return energy / total


## 4 · Chequeo de deriva


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/stats.py:finite_values": "8aa861655f2b",
    "musepipe/stats.py:robust_sigma": "ef2aa72a72de",
    "musepipe/stats.py:robust_sigma_axis0": "0b976272de28",
    "musepipe/apertures.py:angular_separation_deg": "ce3b83206746",
    "musepipe/apertures.py:aperture_weights": "d8e1fd8bb88d",
    "musepipe/apertures.py:same_radius_control_positions": "fb89fcf9dd7d",
    "musepipe/extraction/aperture.py:_as_cube": "67ede036d305",
    "musepipe/extraction/aperture.py:_npix_eff": "32ecf15dcce9",
    "musepipe/extraction/aperture.py:aperture_spectrum": "214c68e30b47",
    "musepipe/extraction/aperture.py:annulus_background_spectrum": "68ec4c4e7299",
    "musepipe/extraction/aperture.py:aperture_stat_error": "a72d7d66c6e6",
    "musepipe/extraction/aperture.py:control_aperture_spectra": "d8a9a0f28448",
    "musepipe/extraction/aperture.py:_flag_window": "7ba686789970",
    "musepipe/extraction/aperture.py:channel_flags": "dbceb28a0a0b",
    "musepipe/extraction/aperture.py:aperture_correction_from_psf": "a2d1719a4ffc",
    "musepipe/halosub.py:select_reference_spaxels": "bfb3aeb23aac",
    "musepipe/halosub.py:reference_spectrum": "b1e5e3857fec",
    "musepipe/halosub.py:safe_reference": "176147b64b00",
    "musepipe/halosub.py:fill_nan_along_axis0": "af1e7d958e4e",
    "musepipe/halosub.py:LpmResult": "4574dae0f49e",
    "musepipe/halosub.py:lpm_design_matrix": "619cd2220508",
    "musepipe/halosub.py:lpm_fit_mask": "f92b8928f6d3",
    "musepipe/halosub.py:lpm_subtract": "3f43c95bb016",
    "musepipe/halosub.py:lpm_coefficient_energy_share": "b11253b944ae"
}

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        node = next((n for n in _ast.parse(text).body
                     if isinstance(n, (_ast.FunctionDef, _ast.ClassDef)) and n.name == name),
                    None)
        if node is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        start = min([node.lineno] + [d.lineno for d in node.decorator_list]) - 1
        src = ''.join(lines[start:node.end_lineno]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera: python scripts/build_debug_notebooks.py --target {TARGET} C6')
else:
    print(f'sin deriva: las {len(_SHAS)} piezas copiadas son las de musepipe')


## 5 · Paso 1 — los spaxels de referencia

El halo se estima **con el propio campo**: se eligen los spaxels cuyo flujo está entre dos percentiles —ni saturados por el núcleo de la estrella ni dominados por el ruido del borde— **excluyendo un disco alrededor del compañero**, para no meterlo en su propia referencia. La spec exige ≥50 spaxels: por debajo de eso no hay diversidad espectral que explotar.


In [ ]:
keep, keep_qc = select_reference_spaxels(
    CUBE, flux_lo_frac=FLUX_LO, flux_hi_frac=FLUX_HI,
    exclude_yx=[OBJECT_YX], exclude_radius_px=EXCLUDE_RADIUS_PX)
print('spaxels conservados:', keep_qc['n_spaxels_kept'],
      '| mínimo que exige la spec: 50')

campo = np.nanmedian(CUBE[::20], axis=0)
pos = campo[np.isfinite(campo) & (campo > 0)]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
im1 = a1.imshow(campo, origin='lower', cmap='magma',
                norm=LogNorm(vmin=np.percentile(pos, 60), vmax=np.percentile(pos, 99.9)))
fig.colorbar(im1, ax=a1, shrink=0.8).set_label('flujo mediano · escala LOG', fontsize=7)
a1.set_title('el campo', fontsize=9)
a2.imshow(campo, origin='lower', cmap='gray',
          norm=LogNorm(vmin=np.percentile(pos, 60), vmax=np.percentile(pos, 99.9)))
a2.imshow(np.where(keep, 1.0, np.nan), origin='lower', cmap='cool', alpha=0.55,
          vmin=0, vmax=1)
a2.plot(OBJECT_YX[1], OBJECT_YX[0], marker='o', ms=9, mfc='none', mec='tab:red', mew=1.5)
a2.annotate('compañero (excluido)', (OBJECT_YX[1], OBJECT_YX[0]),
            textcoords='offset points', xytext=(0, 11), ha='center',
            fontsize=7, color='tab:red')
n_keep = int(keep_qc['n_spaxels_kept'])
a2.set_title(f'spaxels de referencia ({n_keep}) en color', fontsize=9)
for ax in (a1, a2):
    ax.set_xlabel('x [px]')
a1.set_ylabel('y [px]')
fig.tight_layout(); plt.show()


## 6 · Paso 2 — el espectro estelar de referencia

La mediana de los spaxels elegidos, normalizada. Es **el espectro del halo**: lo que los dos métodos van a escalar y restar en cada spaxel.


In [ ]:
s_hat = reference_spectrum(CUBE, keep)
print('referencia: mediana', round(float(np.nanmedian(s_hat)), 4),
      '| canales no finitos:', int((~np.isfinite(s_hat)).sum()))
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(WAVE, s_hat, lw=0.7)
ax.axvline(6563, color='tab:red', ls=':', label='Hα')
ax.set_xlabel('λ [Å]'); ax.set_ylabel('referencia ŝ'); ax.legend(fontsize=8)
ax.set_title('espectro estelar de referencia (mediana de los spaxels elegidos)', fontsize=9)
fig.tight_layout(); plt.show()


## 7 · La sustracción LPM

La matriz de diseño tiene una columna por grado: `P_k(λ̃) · ŝ(λ)`. El ajuste es por mínimos cuadrados sobre los canales **no enmascarados** (fuera de Hα, Hβ, O I…), y el modelo se evalúa después en **todos** los canales — incluidos los de la línea, que es donde se quiere que no haya aprendido nada.

Subir el grado sigue mejor el halo pero se acerca a poder absorber la línea; es la perilla que el diagnóstico de energía por grado vigila.


In [ ]:
mask_fit = lpm_fit_mask(WAVE, s_hat)
res = lpm_subtract(CUBE, WAVE, s_hat, degree=LPM_DEGREE)
residual = res.residual_cube
print('grado', LPM_DEGREE, '| canales usados en el ajuste:', int(mask_fit.sum()),
      f'de {WAVE.size} ({100 * mask_fit.mean():.1f}%)')
print('energía por grado:', np.round(lpm_coefficient_energy_share(res.coeffs), 3))
yc, xc = int(round(OBJECT_YX[0])), int(round(OBJECT_YX[1]))
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
design = lpm_design_matrix(WAVE, s_hat, degree=LPM_DEGREE)
for k in range(design.shape[1]):
    a1.plot(WAVE, design[:, k], lw=0.8, label=f'P{k}(λ̃)·ŝ')
a1.set_ylabel('columnas del diseño'); a1.legend(fontsize=7, ncol=5)
a1.set_title('la base: la referencia modulada por Legendre', fontsize=9)
a2.plot(WAVE, CUBE[:, yc, xc], lw=0.5, color='0.6', label='spaxel del compañero (crudo)')
a2.plot(WAVE, CUBE[:, yc, xc] - residual[:, yc, xc], lw=1.0, color='tab:orange',
        label='halo modelado')
a2.plot(WAVE, residual[:, yc, xc], lw=0.8, color='tab:blue', label='residual')
a2.fill_between(WAVE, *a2.get_ylim(), where=~mask_fit, color='tab:red', alpha=0.10,
                label='canales EXCLUIDOS del ajuste (líneas)')
a2.axvline(6563, color='tab:red', ls=':')
a2.set_xlabel('λ [Å]'); a2.legend(fontsize=8)
fig.tight_layout(); plt.show()


## 8 · Paso 4 — la apertura sobre el residual

A partir de aquí es **exactamente C2**: caja box3 en la posición de B3 sobre el cubo residual, controles al mismo radio procesados igual, σ empírico de su dispersión y `apcorr` de la curva de crecimiento de C1. Por eso estos métodos comparten convención de flujo con el resto.


In [ ]:
APERTURE = {'kind': 'box', 'size': 3}
raw_flux, npix_eff = aperture_spectrum(residual, OBJECT_YX, APERTURE)
if ANNULUS is not None:
    bkg = annulus_background_spectrum(residual, OBJECT_YX, ANNULUS[0], ANNULUS[1],
                                     exclude_yx=STAR_YX,
                                     exclude_radius=(ANNULUS[2] if len(ANNULUS) > 2 else 30.0))
    raw_flux = raw_flux - bkg * npix_eff
controls_yx, control_spectra, control_npix = control_aperture_spectra(
    residual, OBJECT_YX, STAR_YX, APERTURE,
    n_controls=N_CONTROLS, exclude_angle_deg=EXCLUDE_ANGLE_DEG)
raw_err_emp = (robust_sigma_axis0(control_spectra) if control_spectra.shape[0] >= 2
               else np.full(WAVE.size, robust_sigma(raw_flux)))
usable = (STAT_CUBE is not None and str(ERROR_MODE).lower() != 'empirical'
          and STAT_STATUS.lower() != 'red')
# El STAT se propaga SIEMPRE que exista, aunque la etapa no lo use como
# sigma: es la segunda estimación con la que contrastar el empírico.
raw_err_stat = (aperture_stat_error(STAT_CUBE, OBJECT_YX, APERTURE,
                                    stat_factor=STAT_FACTOR,
                                    covariance_factor=COV_FACTOR)
                if STAT_CUBE is not None else None)
raw_err = raw_err_stat if usable else np.asarray(raw_err_emp, float)
apcorr, apcorr_mode, _nr = aperture_correction_from_psf(
    WAVE, APERTURE, PSF_MODEL, center_yx=OBJECT_YX, correction_mode=APCORR_MODE)
flux = raw_flux * apcorr
flux_err_emp = np.asarray(raw_err_emp, float) * apcorr
flux_err_stat = None if raw_err_stat is None else raw_err_stat * apcorr
modo_err = 'stat' if usable else 'empirical'
print(f'{len(controls_yx)} controles | modo error: {modo_err}'
      f' | apcorr mediana {float(np.nanmedian(apcorr)):.1f}')

def binea(wave, flux, err, n):
    n = int(n); corte = (wave.size // n) * n
    w = wave[:corte].reshape(-1, n); f = flux[:corte].reshape(-1, n)
    e = err[:corte].reshape(-1, n)
    bueno = np.isfinite(f) & np.isfinite(e); cuenta = bueno.sum(axis=1)
    with np.errstate(invalid='ignore', divide='ignore'):
        wb = np.nanmean(np.where(bueno, w, np.nan), axis=1)
        fb = np.nansum(np.where(bueno, f, 0.0), axis=1) / np.maximum(cuenta, 1)
        eb = np.sqrt(np.nansum(np.where(bueno, e, 0.0) ** 2, axis=1)) / np.maximum(cuenta, 1)
    fb[cuenta == 0] = np.nan; eb[cuenta == 0] = np.nan
    return wb, fb, eb

BIN_CANALES = 25
N_EFF_OVER_N = 0.43   # medido por G1 (docs/noise_model.md)
wb, fb, eb = binea(WAVE, flux, flux_err_emp, BIN_CANALES)
rojo = (wb >= 7500) & (wb <= 9000)
print(f'flujo mediano en 7500–9000 Å: {np.nanmedian(fb[rojo]):9.2f}'
      f' ± {np.nanmedian(eb[rojo] / np.sqrt(N_EFF_OVER_N)):.2f}')
fig, ax = plt.subplots(figsize=(11, 3.8))
for lo, hi in BAD_WINDOWS_A:
    ax.axvspan(lo, hi, color='0.85', zorder=0)
ax.plot(WAVE, flux, lw=0.3, color='0.75', label='por canal')
ax.errorbar(wb, fb, yerr=eb / np.sqrt(N_EFF_OVER_N), fmt='o', ms=3, lw=0.9,
            color='tab:blue', label=f'binado {BIN_CANALES} ch, ±σ corregido por n_eff')
ax.axhline(0, color='0.5', lw=0.7); ax.axvline(6563, color='tab:red', ls=':', label='Hα')
ax.set_ylim(*np.nanpercentile(fb[np.isfinite(fb)], [1, 99]) * np.array([2.5, 2.5]))
ax.set_xlabel('λ [Å]'); ax.legend(fontsize=8)
ax.set_title('C6 rehecho en el notebook', fontsize=9)
fig.tight_layout(); plt.show()


## 9 · Qué sobrevive a la sustracción

El LPM tampoco distingue «halo» de «compañero»: cualquier componente del compañero **colineal con la referencia modulada** —y su continuo lo es— la absorbe el ajuste (Julo+25 Sect. 4.1). La diferencia con el SGF está en las **líneas**: como Hα está enmascarada, el modelo no puede aprenderla, y abajo se ve que la devuelve entera.

Tres medidas, en este orden:

1. **Cuánto se lleva el modelo** en la box3 del compañero, contra la fotometría de apertura de C2.
2. **Que el modelo es un filtro del propio espectro de la apertura** — no una estimación independiente del halo de la primaria.
3. **La función de respuesta**: qué fracción sobrevive según lo ancha que sea la señal, inyectándola y volviendo a restar con el mismo operador.


In [ ]:
APERTURA_D = {'kind': 'box', 'size': 3}
crudo_b3, _ = aperture_spectrum(CUBE, OBJECT_YX, APERTURA_D)
resid_b3, _ = aperture_spectrum(residual, OBJECT_YX, APERTURA_D)
modelo_b3 = crudo_b3 - resid_b3
fin = np.isfinite(crudo_b3) & np.isfinite(resid_b3)

def _rob(v):
    v = v[np.isfinite(v)]
    m = float(np.median(v))
    return m, float(1.4826 * np.median(np.abs(v - m)))

m_crudo, _ = _rob(crudo_b3[fin])
m_model, _ = _rob(modelo_b3[fin])
m_resid, s_resid = _rob(resid_b3[fin])
print('--- 1 · box3 en el compañero, SIN anillo ni apcorr (unidades del cubo)')
print(f'  suma cruda        {m_crudo:10.2f}')
print(f'  modelo de halo    {m_model:10.2f}   ({100 * m_model / m_crudo:5.1f}% de la cruda)')
print(f'  residual          {m_resid:10.2f}   ({100 * m_resid / m_crudo:5.1f}% de la cruda)'
      f'  ·  σ robusta {s_resid:.2f} → mediana/σ = {m_resid / s_resid:+.2f}')

# Contra el producto de C2, que es la misma box3 con anillo y apcorr.
_p_ap = SD / 'spec_aperture_object.fits'
if _p_ap.exists():
    from musepipe.extraction.product import SpectrumProduct as _SP
    f_ap = np.asarray(_SP.read(_p_ap).flux, float)
    banda = np.isfinite(f_ap) & np.isfinite(flux) & (WAVE >= 7500) & (WAVE <= 9000)
    print(f'\n  continuo 7500–9000 Å · C2 apertura {np.median(f_ap[banda]):9.2f}'
          f'  ·  lpm {np.median(flux[banda]):9.2f}'
          f'  ({100 * np.median(flux[banda]) / np.median(f_ap[banda]):+.1f}%)')
else:
    print('\n  (sin spec_aperture_object.fits: no hay con qué comparar C2)')


In [ ]:
# El operador del método sobre UN espectro, no sobre el cubo: la misma
# función copiada, con un «cubo» de un solo spaxel. Sirve porque el ajuste
# es lineal en los datos con `s_hat` fijo. Se le pasa la máscara del ajuste
# del cubo: con un solo spaxel, `lpm_subtract` se salta la higiene de
# canales (`min_channel_finite_frac`) y el operador no sería el mismo.
def resta_1d(esp):
    return lpm_subtract(esp[:, None, None], WAVE, s_hat, degree=LPM_DEGREE,
                        fit_mask=res.fit_mask).residual_cube[:, 0, 0]

print('--- 2 · el modelo, ¿es un filtro del propio espectro de la apertura?')
# El operador es lineal en los datos con `s_hat` fijo: si el modelo de la
# box3 fuera exactamente eso, aplicarlo a la SUMA de los 9 spaxels tiene
# que dar la SUMA de los 9 modelos. Se cumple canal a canal EXACTO salvo
# donde algún spaxel tiene NaN: ahí el relleno de huecos interpola por
# spaxel, que ya no es lineal, y en el SGF la ventana reparte el estropicio
# ±window/2 canales alrededor. Por eso se mira la distribución de |Δ| y no
# solo el máximo, que lo fijan esos pocos canales.
modelo_1d = crudo_b3 - resta_1d(np.where(fin, crudo_b3, np.nanmedian(crudo_b3)))
cmp_ = fin & np.isfinite(modelo_1d)
dif = np.abs(modelo_b3 - modelo_1d)[cmp_]
esc_m = float(np.median(np.abs(modelo_b3[cmp_])))
print(f'  corr(modelo box3, operador aplicado al espectro crudo) ='
      f' {np.corrcoef(modelo_b3[cmp_], modelo_1d[cmp_])[0, 1]:.6f}')
print(f'  |Δ| sobre una escala de {esc_m:.1f}:'
      f'  mediana {np.median(dif):.2e}  p90 {np.percentile(dif, 90):.2e}'
      f'  p99 {np.percentile(dif, 99):.2f}  máx {dif.max():.2f}')
_malos = int((dif > 0.01 * esc_m).sum())
print(f'  canales que se apartan más del 1%: {_malos} de {cmp_.sum()}'
      f' ({100 * _malos / cmp_.sum():.1f}%) — arrastre de los spaxels no finitos')
print('  → el «halo» de este método en la apertura del compañero sale del espectro')
print('    de la propia apertura, no de una estimación aparte de la primaria.')

print('\n--- 3 · función de respuesta: se inyecta una gaussiana en Hα y se vuelve a restar')
DL = float(np.nanmedian(np.diff(WAVE)))
base = np.where(fin, crudo_b3, np.nanmedian(crudo_b3))
escala = float(np.nanmedian(np.abs(crudo_b3)))
r0 = resta_1d(base)
ANCHOS_A = [2.6, 5.0, 20.0, 60.0]
filas, curvas = [], []
for fwhm in ANCHOS_A:
    sig = fwhm / 2.3548
    linea = np.exp(-0.5 * ((WAVE - 6562.8) / sig) ** 2) * escala
    sale = resta_1d(base + linea) - r0
    ok = np.isfinite(sale) & np.isfinite(linea)
    loc = ok & (np.abs(WAVE - 6562.8) <= 3 * sig)
    filas.append((fwhm, fwhm / DL, np.nanmax(sale) / linea.max(),
                  sale[loc].sum() / linea[loc].sum(), sale[ok].sum() / linea[ok].sum()))
    curvas.append((fwhm, linea, sale))
# Y un continuo plano: es el caso «infinitamente ancho».
plano = resta_1d(base + escala) - r0
frac_cont = float(np.nanmedian(plano[np.isfinite(plano)])) / escala

print(f'  {"FWHM":>8s} {"canales":>8s} {"pico":>8s} {"flujo ±3σ":>11s} {"flujo total":>12s}')
for fwhm, nch, pico, floc, ftot in filas:
    print(f'  {fwhm:8.1f} {nch:8.1f} {100 * pico:7.1f}% {100 * floc:10.1f}%'
          f' {100 * ftot:11.1f}%')
print(f'  {"continuo":>8s} {"—":>8s} {100 * frac_cont:7.2f}%'
      f' {"—":>11s} {"—":>12s}   ← se anula por construcción')

# El throughput que E4 midió de verdad, inyectando en el cubo y re-extrayendo.
_p_h04 = SD / 'stage_h04_qc.json'
if _p_h04.exists():
    _thr = json.loads(_p_h04.read_text(encoding='utf-8')).get(
        'throughput', {}).get('per_method_at_snr5', {})
    if _thr:
        print('\n  throughput medido por E4 a SNR=5 (inyección-recuperación completa):')
        for _m, _v in _thr.items():
            marca = '  ←' if _m == 'lpm' else ''
            print(f'    {_m:16s} {_v.get("throughput", float("nan")):.3f}'
                  f' ± {_v.get("err", float("nan")):.3f}{marca}')

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
for fwhm, linea, sale in curvas:
    a1.plot(WAVE, sale / escala, lw=0.9, label=f'FWHM {fwhm:g} Å')
a1.plot(WAVE, curvas[0][1] / escala, lw=0.8, ls=':', color='0.5', label='inyectada (2.6 Å)')
a1.set_xlim(6562.8 - 120, 6562.8 + 120); a1.axhline(0, color='0.7', lw=0.6)
a1.set_xlabel('λ [Å]'); a1.set_ylabel('lo que sale / escala')
a1.legend(fontsize=7)
a1.set_title('lo que sobrevive de una línea inyectada', fontsize=9)
a2.semilogx([f[0] for f in filas], [100 * f[2] for f in filas], 'o-', label='pico')
a2.semilogx([f[0] for f in filas], [100 * f[4] for f in filas], 's--', label='flujo total')
a2.axhline(100 * frac_cont, color='tab:red', ls=':', label='continuo plano')
a2.axhline(100, color='0.7', lw=0.6); a2.set_ylim(-10, 115)
a2.set_xlabel('FWHM de la señal [Å]'); a2.set_ylabel('sobrevive [%]')
a2.legend(fontsize=7)
a2.set_title('función de respuesta del método', fontsize=9)
fig.tight_layout(); plt.show()


## 10 · Comparación con la cadena

Contra `spec_lpm_object.fits`. Con las perillas por defecto debe salir idéntico; si cambias la ventana del filtro, el grado o la máscara de referencia, aquí se ve cuánto se movió.


In [ ]:
from musepipe.extraction.product import SpectrumProduct
from musepipe.spectral import median_filter_1d

cadena = SpectrumProduct.read(SD / 'spec_lpm_object.fits')
ref_flux = np.asarray(cadena.flux, float)
ok = True
for clave, a, b in (('flujo', flux, ref_flux),
                    ('apcorr', apcorr, np.asarray(cadena.apcorr, float))):
    fin = np.isfinite(a) & np.isfinite(b)
    ig = np.isclose(a[fin], b[fin], rtol=1e-9, atol=0.0)
    print(f'  {clave:7s} idénticos {100 * ig.mean():6.2f}% de {fin.sum()} canales'
          f' | máx |Δ| = {np.abs(a - b)[fin].max():.3e}')
    ok &= bool(ig.all())
print()
print('IDÉNTICO: la copia reproduce la cadena.' if ok else
      'DIFIERE — si has tocado una perilla, es lo esperado; si no, revisa el chequeo de deriva.')

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1]})
a1.plot(WAVE, median_filter_1d(ref_flux, 41), lw=1.6, color='0.6', label='cadena')
a1.plot(WAVE, median_filter_1d(flux, 41), lw=1.0, ls='--', color='tab:blue',
        label='este notebook')
a1.legend(fontsize=8); a1.set_ylabel('flujo (mediana 41 ch)')
a2.plot(WAVE, flux - ref_flux, lw=0.7, color='tab:purple')
a2.axhline(0, color='0.7', lw=0.6)
a2.set_ylabel('este − cadena'); a2.set_xlabel('λ [Å]')
fig.tight_layout(); plt.show()


## 11 · Figura de paper y tabla

Las figuras anteriores son de diagnóstico. Ésta es la que se publica, y por eso cambia en tres cosas:

- **Sin binar**: cada canal con su σ. Binar es cómodo para leer un continuo, pero esconde justo lo que se quiere enseñar (o no enseñar): que en Hα no hay nada por encima del ruido **a la resolución del dato**.
- **Dos barras de error**: la **empírica** (dispersión de los controles procesados igual que el objeto) como banda, y la **propagada del STAT** como línea. Que se vean las dos es la forma honesta de enseñar que el STAT del cubo no es σ ([`docs/noise_model.md`](../docs/noise_model.md)).
- **Marcado completo**: las **bandas telúricas** sombreadas por especie (O₂ naranja, H₂O cian) con la **transmisión medida esa noche** en la tira de arriba, las **líneas de acreción** por familia (Balmer, He I, prohibidas, O I, Ca II, Paschen) y las **líneas de emisión de cielo** en gris discontinuo.

### Por qué bandas telúricas y no líneas telúricas

A la resolución de MUSE (FWHM ≈ 2.5 Å) las líneas individuales de O₂ y H₂O **no se resuelven**: dentro de un píxel espectral caen muchas. Marcar líneas sueltas daría una precisión que el dato no tiene, así que se marcan **bandas**. `molecfit` no está disponible aquí y, en estos datos, **no convergió** (A3 corrigió con la estrella telúrica estándar), pero de ahí quedó una **curva de transmisión medida** en la misma rejilla de λ: eso es más específico que cualquier lista de laboratorio y es lo que se dibuja. Catálogo y curva: [`musepipe/telluric_lines.py`](../musepipe/telluric_lines.py).

### Y sus datos, en columnas

La celda **escribe la tabla** además de la figura, en **ECSV** (el estándar portable de astropy): texto plano, con las unidades y la procedencia en la cabecera, que se lee con `Table.read(ruta)` sin configurar nada y se puede mandar por correo. Una figura sin sus datos no es un resultado citable.

> Sale de los números recalculados aquí, no del producto de la cadena: si has tocado una perilla, la figura y la tabla la llevan. Ficheros con sufijo `_debug`, que no pisan lo que exporta el notebook de auditoría.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    ROOT_P = ROOT
    METHOD_P = 'lpm'
    PRODUCT_P = 'recalculado en C6_lpm_debug'
    TARGET_P = str(TARGET).replace(' ', '') + '_debug'
    BUNIT_P = BUNIT or 'ADU'
    W_P, F_P = WAVE, flux
    E_P, E_ALT_P = flux_err_emp, flux_err_stat
    EXTRA_P = {'flux_err_stat': flux_err_stat, 'apcorr': apcorr,
               'npix_eff': npix_eff,
               'flags': channel_flags(WAVE, bad_windows_A=BAD_WINDOWS_A)}
    MODO_P = modo_err
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'lpm, rehecho en el notebook',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c6_lpm_debug'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper' + '.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / ('spectrum_paper' + '.pdf'))}))
    print('figura ->', outdir / ('spectrum_paper' + '.pdf'))
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## La primaria, en la misma figura

Cierra el notebook el espectro de la **estrella central**, dibujado con **exactamente la misma figura** que el del compañero: mismos tramos, mismas bandas telúricas, mismas líneas marcadas y la misma tira de transmisión. Puestas una al lado de otra se comparan sin trampa.

Para qué sirve mirarla:

- **Es la referencia del halo.** Todo lo que este notebook resta —anillo, modelo de PSF, referencia estelar— sale de esta fuente. Su forma es la del fondo que hay que quitar, y su color explica por qué el halo es más brillante en el rojo.
- **Separa lo atmosférico de lo del objeto.** Las bandas telúricas y los residuos de cielo aparecen en las dos, y con la misma λ. Un rasgo que solo esté en el compañero es del compañero; uno que esté en las dos, no.
- **Da la escala.** La primaria es unas mil veces más brillante, así que cualquier fracción de su luz que se cuele en la ventana del compañero pesa mucho.

> Sale del **producto de la cadena** (`spec_psffit_star.fits`, que escribe C4), no de un recálculo de este notebook: así es la misma primaria en los cinco notebooks de análisis y sirve de referencia común. Si C4 no se ha ejecutado para este objeto, la celda lo dice y sigue.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    from astropy.io import fits
    ROOT_P = nb.project_root()
    METHOD_P = 'psffit_star'
    PRODUCT_P = 'spec_psffit_star.fits'
    TARGET_P = (nb.run_target(RUN_ID) or RUN_ID).replace(' ', '')
    _h = fits.open(nb.run_dir(RUN_ID) / 'stages' / PRODUCT_P)
    _d = _h[1].data
    _cols = list(_d.columns.names)
    # La unidad viaja con el dato (BUNIT); no hay default silencioso.
    BUNIT_P = _h[1].header.get('BUNIT') or 'ADU'
    W_P = np.asarray(_d['wave_A'], float)
    F_P = np.asarray(_d['flux'], float)
    # El empírico manda; `flux_err` es el que eligió la etapa y solo
    # aporta algo cuando NO es el empírico (ver la nota de abajo).
    E_P = np.asarray(_d['flux_err_emp' if 'flux_err_emp' in _cols
                        else 'flux_err'], float)
    E_ALT_P = np.asarray(_d['flux_err'], float)
    EXTRA_P = {'flux_err_stat': E_ALT_P}
    for _c in ('apcorr', 'npix_eff', 'flags'):
        if _c in _cols:
            EXTRA_P[_c] = np.asarray(_d[_c])
    _h.close()
    try:
        MODO_P = (nb.load_qc('stages/spec_psffit_qc.json', RUN_ID).get('errors') or {}).get('mode')
    except Exception:
        MODO_P = None
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'espectro de la PRIMARIA (producto de C4, referencia)',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c6_lpm_debug'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper_star_ref' + '.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / ('spectrum_paper_star_ref' + '.pdf'))}))
    print('figura ->', outdir / ('spectrum_paper_star_ref' + '.pdf'))
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)
